In [0]:

## Spark Performance Settings
spark.conf.set("spark.sql.shuffle.partitions", "auto")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

In [0]:
dbutils.widgets.text(name="lookback_period", defaultValue="30", label="lookback_period")
dbutils.widgets.text(name="catalog_schema_prefix", defaultValue="marketingdata_dev.claire_wilsonbarnes", label="catalog_schema_prefix")


In [0]:

%sql

CREATE OR REPLACE TABLE IDENTIFIER(:catalog_schema_prefix || '.pctr_training_ctr_summary') AS (
SELECT 
      d.rundate
    , 'Overall' AS split_type
    , 'All' AS feature_name
    -- , a.control_sheet_AdID
    , a.title 
    , a.campaign
    , a.versionnumber 
    , a.algodivision
    -- Take overall CTR for all ads in period if unknown 
    , SUM(a.number_clicks)/ SUM(a.number_impressions) AS ctr
    , SUM(a.number_impressions) AS num_impressions
    , SUM(a.number_clicks) AS num_clicks
FROM 
    IDENTIFIER(:catalog_schema_prefix || '.pctr_training_dates') AS d
    INNER JOIN IDENTIFIER(:catalog_schema_prefix || '.pctr_training_clicks_lookback') AS a
        ON a.date BETWEEN d.rundate - (INTERVAL '1 DAY' * (:lookback_period + 1)) AND d.rundate - INTERVAL '1' DAY
GROUP BY    
      d.rundate
    -- , a.control_sheet_AdID
    , a.title 
    , a.campaign
    , a.versionnumber 
    , a.algodivision
    , feature_name
    , split_type 
--Device Type
UNION 
SELECT 
    d.rundate
    , 'Device' AS split_type
    , a.device_simple AS feature_name
    -- , a.control_sheet_AdID
    , a.title 
    , a.campaign
    , a.versionnumber 
    , a.algodivision
    , SUM(number_clicks)/SUM(number_impressions) AS ctr 
    , SUM(a.number_impressions) AS num_impressions
    , SUM(a.number_clicks) AS num_clicks
FROM 
    IDENTIFIER(:catalog_schema_prefix || '.pctr_training_dates')  AS d
    INNER JOIN IDENTIFIER(:catalog_schema_prefix || '.pctr_training_clicks_lookback') AS a
        ON a.date BETWEEN d.rundate - (INTERVAL '1 DAY' * (:lookback_period + 1)) AND d.rundate - INTERVAL '1' DAY
GROUP BY    
      device_simple
    --  , a.control_sheet_AdID
     , a.title 
    , a.campaign
    , a.versionnumber 
    , a.algodivision
    , split_type
    , d.rundate

UNION
--Channel ctr
SELECT 
      d.rundate
    , 'Channel' AS split_type
    , channel_simple AS feature_name
    -- , a.control_sheet_AdID
    , a.title 
    , a.campaign
    , a.versionnumber 
    , a.algodivision
    , SUM(number_clicks)/SUM(number_impressions)AS ctr 
    , SUM(a.number_impressions) AS num_impressions
    , SUM(a.number_clicks) AS num_clicks
FROM 
    IDENTIFIER(:catalog_schema_prefix || '.pctr_training_dates')  AS d
    INNER JOIN IDENTIFIER(:catalog_schema_prefix || '.pctr_training_clicks_lookback') AS a
        ON a.date BETWEEN d.rundate - (INTERVAL '1 DAY' * (:lookback_period + 1)) AND d.rundate - INTERVAL '1' DAY
GROUP BY    
      split_type
    , feature_name
    -- , a.control_sheet_AdID
    , a.title 
    , a.campaign
    , a.versionnumber 
    , a.algodivision
    , d.rundate
UNION
-- Geo-clicks 
SELECT 
     d.rundate
    , 'GeoCountry' AS split_type
     , geocountry_simple AS feature_name
    -- , a.control_sheet_AdID
    , a.title 
    , a.campaign
    , a.versionnumber 
    , a.algodivision
    , SUM(number_clicks)/SUM(number_impressions) AS ctr 
    , SUM(a.number_impressions) AS num_impressions
    , SUM(a.number_clicks) AS num_clicks
FROM 
    IDENTIFIER(:catalog_schema_prefix || '.pctr_training_dates')  AS d
    INNER JOIN IDENTIFIER(:catalog_schema_prefix || '.pctr_training_clicks_lookback') AS a
        ON a.date BETWEEN d.rundate - (INTERVAL '1 DAY' * (:lookback_period + 1)) AND d.rundate - INTERVAL '1' DAY

GROUP BY    
     split_type
    , feature_name
    -- , a.control_sheet_AdID
    , a.title 
    , a.campaign
    , a.versionnumber 
    , a.algodivision
    , d.rundate

UNION 
    -- DOW 
SELECT 
    d.rundate
    , 'DOW' AS split_type
    , dow AS feature_name
    -- , a.control_sheet_AdID
    , a.title 
    , a.campaign
    , a.versionnumber 
    , a.algodivision
    , SUM(number_clicks)/SUM(number_impressions) AS ctr 
    , SUM(a.number_impressions) AS num_impressions
    , SUM(a.number_clicks) AS num_clicks
FROM 
    IDENTIFIER(:catalog_schema_prefix || '.pctr_training_dates')  AS d
    INNER JOIN IDENTIFIER(:catalog_schema_prefix || '.pctr_training_clicks_lookback') AS a
        ON a.date BETWEEN d.rundate - (INTERVAL '1 DAY' * (:lookback_period + 1)) AND d.rundate - INTERVAL '1' DAY 
GROUP BY    
     split_type
    , feature_name
    -- , a.control_sheet_AdID
    , a.title 
    , a.campaign
    , a.versionnumber 
    , a.algodivision
    , d.rundate
UNION 
-- Gender 
SELECT 
    d.rundate
    , 'Gender' AS split_type
     , gender AS feature_name
    -- , a.control_sheet_AdID
    , a.title 
    , a.campaign
    , a.versionnumber 
    , a.algodivision
    , SUM(number_clicks)/SUM(number_impressions) AS ctr 
    , SUM(a.number_impressions) AS num_impressions
    , SUM(a.number_clicks) AS num_clicks
FROM 
    IDENTIFIER(:catalog_schema_prefix || '.pctr_training_dates')  AS d
    INNER JOIN IDENTIFIER(:catalog_schema_prefix || '.pctr_training_clicks_lookback') AS a
        ON a.date BETWEEN d.rundate - (INTERVAL '1 DAY' * (:lookback_period + 1)) AND d.rundate - INTERVAL '1' DAY

GROUP BY    
    d.rundate
    , split_type
    , feature_name
    -- , a.control_sheet_AdID
    , a.title 
    , a.campaign
    , a.versionnumber 
    , a.algodivision
);

In [0]:
%sql
CREATE OR REPLACE TABLE IDENTIFIER(:catalog_schema_prefix || '.pctr_training_ctr_imputation') AS  (
SELECT 
    split_type
    ,feature_name 
    , algodivision
    , PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY ctr) AS med_ctr
    , PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY num_impressions) AS med_impressions
    , rundate
FROM
    IDENTIFIER(:catalog_schema_prefix || '.pctr_training_ctr_summary') 
GROUP BY 
      split_type
    , feature_name 
    , algodivision
    , rundate
);
